# Qwen3-0.6B Ranker Ã¢â¬â DLIS Deploy Preparation

Run cells in order to:
1. Install dependencies
2. Verify checkpoint
3. Test inference locally
4. Build Docker image & push to ACR

## 1. Install Dependencies

In [ ]:
!pip install -q vllm fastapi uvicorn[standard] fire pydantic transformers numpy sentencepiece

## 2. Verify Checkpoint

In [ ]:
import os, json

CKPT = '../output/v10_Qwen3-0.6B_all_ep2/checkpoint-2848'
DEPLOY_DIR = '.'

# Check checkpoint files
required = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
for f in required:
    path = os.path.join(CKPT, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{'✅' if exists else '❌'} {f}: {size/1024/1024:.1f} MB" if exists else f"❌ {f}: MISSING")

# Show model config
with open(os.path.join(CKPT, 'config.json')) as f:
    cfg = json.load(f)
print(f"\nModel: {cfg['architectures'][0]}")
print(f"Hidden: {cfg['hidden_size']}, Layers: {cfg['num_hidden_layers']}, Heads: {cfg['num_attention_heads']}")
print(f"Dtype: {cfg['torch_dtype']}, Vocab: {cfg['vocab_size']}")

## 3. Quick Smoke Test (vLLM engine + score 1 sample)

In [ ]:
import time
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print('Loading vLLM engine...')
llm = LLM(
    model=CKPT,
    dtype='bfloat16',
    trust_remote_code=True,
    max_model_len=4096,
    gpu_memory_utilization=0.9,
)

tokenizer = AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Yes/No token IDs
yes_id = tokenizer.encode(' Yes', add_special_tokens=False)[0]
no_id = tokenizer.encode(' No', add_special_tokens=False)[0]
print(f'Yes token: {yes_id}, No token: {no_id}')
print('vLLM engine ready!')

In [ ]:
import math

# Score a test candidate
SYS = ('I am a recommendation assistant. I read the user\'s interests, recent '
       'conversations, and shown cards, then predict whether they will click '
       'the candidate item. I answer Yes or No.')

body = '''USER_INTERESTS:
- Artificial Intelligence  [classification=topic; strength=0.95; status=stable]
- Electric Vehicles  [classification=topic; strength=0.72; status=emerging]

SHOWN_CARDS:
- GPT-5 Released with Multimodal Capabilities
- Tesla Q1 Earnings Beat Expectations

CANDIDATE:
Title: OpenAI Announces New Reasoning Model
Summary: OpenAI has released a new model focused on complex reasoning tasks.
Matched Interest: Artificial Intelligence

Will the user click this candidate? Answer Yes or No.'''

msgs = [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': body}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

t0 = time.time()
sampling_params = SamplingParams(max_tokens=1, temperature=0, logprobs=100)
outputs = llm.generate([text], sampling_params)

logprobs_dict = outputs[0].outputs[0].logprobs[0]
yes_lp = logprobs_dict[yes_id].logprob if yes_id in logprobs_dict else -100.0
no_lp = logprobs_dict[no_id].logprob if no_id in logprobs_dict else -100.0
max_lp = max(yes_lp, no_lp)
p_yes = math.exp(yes_lp - max_lp) / (math.exp(yes_lp - max_lp) + math.exp(no_lp - max_lp))
ms = (time.time() - t0) * 1000

print(f'P(click) = {p_yes:.4f}  ({ms:.0f}ms, vLLM)')
print('✅ vLLM inference working!' if p_yes > 0.5 else '⚠️  Low score — check prompt format')

In [ ]:
import gc, torch
# Clean up GPU memory
del llm
gc.collect()
torch.cuda.empty_cache()
print('GPU memory freed')

## 4. Build Docker Image & Push to ACR

Option A: Bake checkpoint into image (simpler, larger image ~1.5GB)  
Option B: Mount checkpoint at runtime (smaller image, need volume mount)

In [ ]:
# Option A: Bake checkpoint into image
ACR = 'f9309c3acdd842848c88032e1ec736d2'
IMAGE = 'qwen3-06b-ranker'
TAG = 'v10-vllm'

import shutil
# Copy checkpoint into deploy dir for Docker build
dst = os.path.join(DEPLOY_DIR, 'model')
if not os.path.exists(dst):
    print(f'Copying checkpoint to {dst}...')
    shutil.copytree(CKPT, dst, ignore=shutil.ignore_patterns('optimizer.pt', 'scheduler.pt', 'rng_state_*', 'trainer_state.json', 'training_args.bin'))
    print('Done! (skipped optimizer/scheduler/rng to save space)')
else:
    print(f'Checkpoint already at {dst}')

# Show size
total = sum(os.path.getsize(os.path.join(dst, f)) for f in os.listdir(dst) if os.path.isfile(os.path.join(dst, f)))
print(f'Checkpoint size: {total/1024/1024:.0f} MB')

In [ ]:
# Update Dockerfile to bake model into image
dockerfile_content = """FROM nvidia/cuda:12.4.1-devel-ubuntu22.04

RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3 python3-pip python3-venv python3-dev git && \\
    rm -rf /var/lib/apt/lists/*

WORKDIR /app

COPY requirements.txt .
RUN pip3 install --no-cache-dir -r requirements.txt

COPY inference_server.py .
COPY model/ /model/

EXPOSE 8080

HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python3 -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')"

ENTRYPOINT ["python3", "inference_server.py"]
CMD ["--model_path", "/model", "--port", "8080", "--max_len", "4096", "--tp", "1"]
"""

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)
print('✅ Dockerfile written (vLLM + baked-in v10 0.6B checkpoint)')

In [ ]:
# Login to ACR and build+push (remote build, no local docker needed)
!az acr login --name {ACR}
!az acr build --registry {ACR} --image {IMAGE}:{TAG} --file Dockerfile .
print(f'\n✅ Image pushed: {ACR}.azurecr.io/{IMAGE}:{TAG}')

## 5. Next Steps

1. **AML → DLIS conversion**: Run ADO pipeline `IFF-Deployment_Deploy` with the pushed image
2. **DLIS deploy**: Update deploy pipeline — application=`qwen3-ranker`, 1 GPU (vLLM manages memory via PagedAttention)
3. **Multi-GPU**: For larger models, set `--tp 2` (tensor parallel) in CMD
4. **Test endpoint**:
```bash
curl -X POST https://fabricrouter-external.ingress-dlis.ingress.cus.microsoft-falcon.net/dlis-coreranker.qwen3-ranker/score \
  -H 'Content-Type: application/json' \
  -d @test_request.json
```